# Warmup and gradient clipping test (wp7), Colab, Run all

Two runs in one session, same data, same config except the recipe flags:
- **Run A, recipe:** `--warmup-steps 20 --clip-grad-norm 1.0`, seed 0. Compare against the existing 0.299 run (seed 0, no recipe).
- **Run B, control:** no recipe, seed 1. Shows how much two runs with the same settings differ.

Runtime -> Change runtime type -> T4 (or A100 if available). Then Runtime -> Run all. Keep the tab open. Each run takes about 35 minutes. The last cell zips both results for download.

In [ ]:
# Data: labeled shards from the Hugging Face mirror, about 8 minutes. Asserts split sizes.
import os, subprocess, concurrent.futures as cf, tarfile, pathlib
BASE = "https://huggingface.co/datasets/ferpb/spectralwaste-segmentation/resolve/main/data"
ROOT = pathlib.Path("/content/sw_data"); TMP = ROOT / "_tar"; TMP.mkdir(parents=True, exist_ok=True)
MAP = {"train": "train", "validation": "val", "test": "test"}
SHARDS = [("train", f"train-{i:06d}") for i in range(16)] + \
         [("validation", f"validation-{i:06d}") for i in range(5)] + \
         [("test", f"test-{i:06d}") for i in range(6)]

def fetch(item):
    src, shard = item; split = MAP[src]; tp = TMP / f"{shard}.tar"
    subprocess.run(["curl", "-s", "-L", "--fail", "--retry", "5", "-C", "-", "-o", str(tp), f"{BASE}/{shard}.tar"], check=True)
    (ROOT / "hyper" / split).mkdir(parents=True, exist_ok=True)
    (ROOT / "labels_hyper_lt" / split).mkdir(parents=True, exist_ok=True)
    with tarfile.open(tp) as tf:
        for m in tf.getmembers():
            b = os.path.basename(m.name)
            if b.endswith(".hsi.tiff"):
                dst = ROOT / "hyper" / split / (b[:-len(".hsi.tiff")] + ".tiff")
            elif b.endswith(".label_hsi_lt.png"):
                dst = ROOT / "labels_hyper_lt" / split / (b[:-len(".label_hsi_lt.png")] + ".png")
            else:
                continue
            with tf.extractfile(m) as s, open(dst, "wb") as o:
                o.write(s.read())
    tp.unlink()
    return shard

with cf.ThreadPoolExecutor(4) as ex:
    for f in cf.as_completed([ex.submit(fetch, s) for s in SHARDS]):
        print("done", f.result())

counts = {sp: len(list((ROOT / "hyper" / sp).glob("*.tiff"))) for sp in ["train", "val", "test"]}
print(counts)
assert counts == {"train": 514, "val": 167, "test": 171}, "unexpected split sizes, stop and check the download"
print("data OK: 514 / 167 / 171")

In [ ]:
!git clone -q -b wp7-optim-recipe https://github.com/scottn66/255-final-project.git /content/255-final-project
!pip install -q tifffile
!cd /content/255-final-project && git log --oneline -1

## Run A: warmup and clipping, seed 0

In [ ]:
!cd /content/255-final-project && PYTHONPATH=src SW_ROOT=/content/sw_data python -u -m sw.train \
    --model transformer --downsample 2 --epochs 30 --batch-size 4 --num-workers 4 --device cuda \
    --no-cosine --lr 3e-4 --standardize --warmup-steps 20 --clip-grad-norm 1.0 --seed 0 --out-dir /content/sw_runs/wp7_recipe_s0

## Run B: control, no recipe, seed 1

In [ ]:
!cd /content/255-final-project && PYTHONPATH=src SW_ROOT=/content/sw_data python -u -m sw.train \
    --model transformer --downsample 2 --epochs 30 --batch-size 4 --num-workers 4 --device cuda \
    --no-cosine --lr 3e-4 --standardize  --seed 1 --out-dir /content/sw_runs/base_s1

## Compare and download

In [ ]:
import json
def load(p): return [json.loads(l) for l in open(p)]
ref = "/content/sw_runs/base_s1/metrics.jsonl"
A = load("/content/sw_runs/wp7_recipe_s0/metrics.jsonl")
B = load(ref)
print("epoch   recipe_s0  control_s1  recipe grad_norm_max")
for a, b in zip(A, B):
    print(f"{a['epoch']:>5}   {a['val_fg_miou']:.4f}      {b['val_fg_miou']:.4f}      {a.get('grad_norm_max', float('nan')):.2f}")
print("best recipe_s0", round(max(r['val_fg_miou'] for r in A), 4), " best control_s1", round(max(r['val_fg_miou'] for r in B), 4))
print("compare to earlier no-recipe seed-0 run: best 0.2994 at epoch 28")
from google.colab import files
!cd /content/sw_runs && zip -qr wp7_results.zip wp7_recipe_s0 base_s1
files.download("/content/sw_runs/wp7_results.zip")